# U.S. Household Financial Pressure Dashboard

This notebook uses the existing local FRED CSV files in `data/` to answer: **Are U.S. households under financial pressure right now, and which indicators are driving that pressure?**

The dashboard centers on a 7-indicator household financial pressure framework. The headline index averages each indicator's latest normalized pressure percentile, while unemployment and the federal funds rate remain supporting economic context only.


In [1]:
import pandas as pd
import altair as alt
from IPython.display import display
from pathlib import Path

alt.data_transformers.disable_max_rows()

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "html_charts"
OUTPUT_DIR.mkdir(exist_ok=True)

SERIES_CONFIG = {
    "TDSP": {"file": "TDSP.csv", "value_col": "TDSP", "indicator": "Debt Service Ratio", "category": "Household pressure indicators", "direction": "higher_is_worse"},
    "PSAVERT": {"file": "PSAVERT.csv", "value_col": "PSAVERT", "indicator": "Personal Saving Rate", "category": "Household pressure indicators", "direction": "lower_is_worse"},
    "DRCCLACBS": {"file": "DRCCLACBS.csv", "value_col": "DRCCLACBS", "indicator": "Credit Card Delinquency Rate", "category": "Household pressure indicators", "direction": "higher_is_worse"},
    "UNRATE": {"file": "UNRATE.csv", "value_col": "UNRATE", "indicator": "Unemployment Rate", "category": "Supporting economic context", "direction": "higher_is_worse"},
    "FEDFUNDS": {"file": "FEDFUNDS.csv", "value_col": "FEDFUNDS", "indicator": "Federal Funds Rate", "category": "Supporting economic context", "direction": "higher_is_worse"},
}

HOUSEHOLD_PRESSURE_INDICATORS = [
    "Debt Service Ratio",
    "Personal Saving Rate",
    "Credit Card Delinquency Rate",
    "CPI YoY Inflation",
    "Real Median Weekly Earnings",
    "Consumer Sentiment",
    "Housing Payment-to-Income Ratio",
]
SUPPORTING_CONTEXT_INDICATORS = ["Unemployment Rate", "Federal Funds Rate"]
INDICATOR_ORDER = HOUSEHOLD_PRESSURE_INDICATORS + SUPPORTING_CONTEXT_INDICATORS
PRESSURE_COLORS = ["#2a9d8f", "#f4a261", "#d95f5f"]
GROUP_COLORS = ["#33658a", "#7a6f9b"]


def clean_local_fred_csv(file_name, value_col):
    df = pd.read_csv(DATA_DIR / file_name)
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    return df.dropna(subset=[value_col]).sort_values("observation_date").reset_index(drop=True)


def clean_configured_series(series_code):
    config = SERIES_CONFIG[series_code]
    return clean_local_fred_csv(config["file"], config["value_col"])


series_data = {series_code: clean_configured_series(series_code) for series_code in SERIES_CONFIG}

data_inventory = pd.DataFrame([
    {
        "Series": config["indicator"],
        "FRED code": series_code,
        "Rows": len(series_data[series_code]),
        "First date": series_data[series_code]["observation_date"].min().strftime("%Y-%m-%d"),
        "Latest date": series_data[series_code]["observation_date"].max().strftime("%Y-%m-%d"),
    }
    for series_code, config in SERIES_CONFIG.items()
])

display(data_inventory)


,Series,FRED code,Rows,First date,Latest date
0,Debt Service Ratio,TDSP,84,2005-01-01,2025-10-01
1,Personal Saving Rate,PSAVERT,808,1959-01-01,2026-04-01
2,Credit Card Delinquency Rate,DRCCLACBS,141,1991-01-01,2026-01-01
3,Unemployment Rate,UNRATE,940,1948-01-01,2026-05-01
4,Federal Funds Rate,FEDFUNDS,863,1954-07-01,2026-05-01


In [2]:
def pressure_status(score):
    if score >= 75:
        return "elevated"
    if score >= 40:
        return "moderate/mixed"
    return "lower"


def direction_label(direction):
    if direction == "higher_is_worse":
        return "Higher values indicate higher pressure"
    if direction == "lower_is_worse":
        return "Lower values indicate higher pressure"
    return "Unknown indicator relationship"


def pressure_percentile_for_values(values, latest_value, direction):
    historical_percentile = (values <= latest_value).mean() * 100
    if direction == "higher_is_worse":
        return historical_percentile
    return 100 - historical_percentile


def format_latest_value(value, unit):
    if unit == "%":
        return f"{value:.2f}%"
    if unit == "real dollars":
        return f"${value:,.0f}"
    if unit == "index":
        return f"{value:.1f} index"
    return f"{value:.2f}"


def summarize_indicator(series_code):
    config = SERIES_CONFIG[series_code]
    value_col = config["value_col"]
    df = series_data[series_code]
    latest_row = df.iloc[-1]
    latest_value = latest_row[value_col]
    values = df[value_col].dropna()
    pressure_percentile = pressure_percentile_for_values(values, latest_value, config["direction"])
    return {
        "series_code": series_code,
        "indicator": config["indicator"],
        "group": config["category"],
        "latest_date": latest_row["observation_date"],
        "latest_value": latest_value,
        "unit": "%",
        "historical_average": values.mean(),
        "historical_median": values.median(),
        "historical_min": values.min(),
        "historical_max": values.max(),
        "pressure_percentile": pressure_percentile,
        "pressure_level": pressure_status(pressure_percentile),
        "pressure_direction": config["direction"],
        "pressure_direction_label": direction_label(config["direction"]),
        "indicator_order": INDICATOR_ORDER.index(config["indicator"]),
    }


def summarize_context_indicator(df, value_col, indicator, group, unit, direction):
    clean_df = df.dropna(subset=[value_col]).sort_values("observation_date")
    latest_row = clean_df.iloc[-1]
    latest_value = latest_row[value_col]
    values = clean_df[value_col].dropna()
    pressure_percentile = pressure_percentile_for_values(values, latest_value, direction)
    return {
        "series_code": None,
        "indicator": indicator,
        "group": group,
        "latest_date": latest_row["observation_date"],
        "latest_value": latest_value,
        "unit": unit,
        "historical_average": values.mean(),
        "historical_median": values.median(),
        "historical_min": values.min(),
        "historical_max": values.max(),
        "pressure_percentile": pressure_percentile,
        "pressure_level": pressure_status(pressure_percentile),
        "pressure_direction": direction,
        "pressure_direction_label": direction_label(direction),
        "indicator_order": INDICATOR_ORDER.index(indicator),
    }


def build_housing_affordability():
    home_prices = clean_local_fred_csv("MSPUS.csv", "MSPUS")
    mortgage_rates = clean_local_fred_csv("MORTGAGE30US.csv", "MORTGAGE30US")
    household_income = clean_local_fred_csv("MEHOINUSA672N.csv", "MEHOINUSA672N")

    home_prices_q = home_prices.set_index("observation_date")[["MSPUS"]].resample("QE").mean()
    mortgage_rates_q = mortgage_rates.set_index("observation_date")[["MORTGAGE30US"]].resample("QE").mean()
    household_income_q = household_income.set_index("observation_date")[["MEHOINUSA672N"]].resample("QE").ffill()

    housing = home_prices_q.join(mortgage_rates_q, how="outer").join(household_income_q, how="outer")
    housing["MEHOINUSA672N"] = housing["MEHOINUSA672N"].ffill()
    housing = housing.dropna(subset=["MSPUS", "MORTGAGE30US", "MEHOINUSA672N"]).reset_index()

    loan_amount = housing["MSPUS"] * 0.80
    monthly_rate = housing["MORTGAGE30US"] / 100 / 12
    number_of_payments = 30 * 12
    housing["estimated_monthly_payment"] = loan_amount * (
        monthly_rate * (1 + monthly_rate) ** number_of_payments
    ) / ((1 + monthly_rate) ** number_of_payments - 1)
    housing["monthly_household_income"] = housing["MEHOINUSA672N"] / 12
    housing["payment_to_income_pct"] = housing["estimated_monthly_payment"] / housing["monthly_household_income"] * 100
    return housing


summary = pd.DataFrame([summarize_indicator(series_code) for series_code in SERIES_CONFIG])
summary = summary.sort_values("indicator_order").reset_index(drop=True)

cpi = clean_local_fred_csv("CPIAUCSL.csv", "CPIAUCSL")
cpi["cpi_yoy"] = cpi["CPIAUCSL"].pct_change(12) * 100
cpi_yoy = cpi.dropna(subset=["cpi_yoy"]).reset_index(drop=True)
real_weekly_earnings = clean_local_fred_csv("LES1252881600Q.csv", "LES1252881600Q")
consumer_sentiment = clean_local_fred_csv("UMCSENT.csv", "UMCSENT")
housing_affordability = build_housing_affordability()

context_pressure_rows = pd.DataFrame([
    summarize_context_indicator(cpi_yoy, "cpi_yoy", "CPI YoY Inflation", "Household pressure indicators", "%", "higher_is_worse"),
    summarize_context_indicator(real_weekly_earnings, "LES1252881600Q", "Real Median Weekly Earnings", "Household pressure indicators", "real dollars", "lower_is_worse"),
    summarize_context_indicator(consumer_sentiment, "UMCSENT", "Consumer Sentiment", "Household pressure indicators", "index", "lower_is_worse"),
    summarize_context_indicator(housing_affordability, "payment_to_income_pct", "Housing Payment-to-Income Ratio", "Household pressure indicators", "%", "higher_is_worse"),
])

all_indicator_pressure_summary = pd.concat([summary, context_pressure_rows], ignore_index=True)
all_indicator_pressure_summary["indicator"] = pd.Categorical(
    all_indicator_pressure_summary["indicator"], categories=INDICATOR_ORDER, ordered=True
)
all_indicator_pressure_summary = all_indicator_pressure_summary.sort_values("indicator").reset_index(drop=True)
all_indicator_pressure_summary["latest_date_label"] = all_indicator_pressure_summary["latest_date"].dt.strftime("%Y-%m-%d")
all_indicator_pressure_summary["latest_value_label"] = all_indicator_pressure_summary.apply(
    lambda row: format_latest_value(row["latest_value"], row["unit"]), axis=1
)
all_indicator_pressure_summary["pressure_percentile_label"] = all_indicator_pressure_summary["pressure_percentile"].map(lambda value: f"{value:.1f}")

household_pressure_summary = all_indicator_pressure_summary[
    all_indicator_pressure_summary["indicator"].isin(HOUSEHOLD_PRESSURE_INDICATORS)
].copy()
household_pressure_summary["indicator"] = pd.Categorical(
    household_pressure_summary["indicator"], categories=HOUSEHOLD_PRESSURE_INDICATORS, ordered=True
)
household_pressure_summary = household_pressure_summary.sort_values("indicator").reset_index(drop=True)

supporting_pressure_summary = all_indicator_pressure_summary[
    all_indicator_pressure_summary["indicator"].isin(SUPPORTING_CONTEXT_INDICATORS)
].copy()

overall_pressure_score = household_pressure_summary["pressure_percentile"].mean()
overall_status = pressure_status(overall_pressure_score)

scorecard_display = pd.concat([
    pd.DataFrame([{
        "Metric": "Overall household financial pressure index",
        "Latest date": "Latest available FRED observations",
        "Latest value": f"{overall_pressure_score:.1f} / 100",
        "Pressure percentile": f"{overall_pressure_score:.1f}",
        "Status": overall_status,
    }]),
    household_pressure_summary.assign(
        Metric=household_pressure_summary["indicator"].astype(str),
        **{
            "Latest date": household_pressure_summary["latest_date_label"],
            "Latest value": household_pressure_summary["latest_value_label"],
            "Pressure percentile": household_pressure_summary["pressure_percentile_label"],
            "Status": household_pressure_summary["pressure_level"],
        },
    )[["Metric", "Latest date", "Latest value", "Pressure percentile", "Status"]],
], ignore_index=True)

indicator_descriptions = {
    "Debt Service Ratio": "Share of disposable income going to required household debt payments.",
    "Personal Saving Rate": "How much income households save after spending and taxes; lower saving means more pressure.",
    "Credit Card Delinquency Rate": "Share of credit card loans where borrowers are falling behind.",
    "CPI YoY Inflation": "Year-over-year cost-of-living pressure.",
    "Real Median Weekly Earnings": "Inflation-adjusted worker earnings; lower real earnings mean more pressure.",
    "Consumer Sentiment": "Household confidence about the economy; lower sentiment signals broader strain.",
    "Housing Payment-to-Income Ratio": "Estimated mortgage payment burden relative to household income.",
}
household_pressure_summary["description"] = household_pressure_summary["indicator"].astype(str).map(indicator_descriptions)


def pressure_insight(label, row):
    return {
        "Card label": label,
        "Indicator": str(row["indicator"]),
        "Description": indicator_descriptions[str(row["indicator"])],
        "Latest value": row["latest_value_label"],
        "Pressure score": f"{row['pressure_percentile']:.1f} / 100",
        "Latest observation": row["latest_date_label"],
        "pressure_percentile": row["pressure_percentile"],
    }

biggest_pressure_driver = household_pressure_summary.loc[household_pressure_summary["pressure_percentile"].idxmax()]
selected_insight_indicators = {str(biggest_pressure_driver["indicator"])}

broadest_strain_candidates = household_pressure_summary[
    household_pressure_summary["indicator"].isin(["Consumer Sentiment", "Housing Payment-to-Income Ratio"])
]
distinct_broadest_strain_candidates = broadest_strain_candidates[
    ~broadest_strain_candidates["indicator"].astype(str).isin(selected_insight_indicators)
]
if distinct_broadest_strain_candidates.empty:
    distinct_broadest_strain_candidates = broadest_strain_candidates
broadest_household_strain_signal = distinct_broadest_strain_candidates.loc[
    distinct_broadest_strain_candidates["pressure_percentile"].idxmax()
]
selected_insight_indicators.add(str(broadest_household_strain_signal["indicator"]))

lowest_pressure_candidates = household_pressure_summary[
    ~household_pressure_summary["indicator"].astype(str).isin(selected_insight_indicators)
]
if lowest_pressure_candidates.empty:
    lowest_pressure_candidates = household_pressure_summary
lowest_current_pressure_signal = lowest_pressure_candidates.loc[lowest_pressure_candidates["pressure_percentile"].idxmin()]

pressure_insight_cards = pd.DataFrame([
    pressure_insight("Biggest pressure driver", biggest_pressure_driver),
    pressure_insight("Broadest household strain signal", broadest_household_strain_signal),
    pressure_insight("Lowest current pressure signal", lowest_current_pressure_signal),
])


/var/folders/rl/07b6fnln0cz697hvlhbgvtsc0000gn/T/ipykernel_13462/1916117030.py:141: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  household_pressure_summary["indicator"] = pd.Categorical(


## 1. Overview / Scorecard

The scorecard averages the normalized pressure scores for the 7 household pressure indicators listed in `HOUSEHOLD_PRESSURE_INDICATORS`. The score uses each indicator's latest available FRED observation, so individual dates differ by release schedule.


In [3]:
display(scorecard_display.rename(columns={"Pressure percentile": "Normalized pressure score"}))


,Metric,Latest date,Latest value,Normalized pressure score,Status
0,Overall household financial pressure index,Latest available FRED observations,60.2 / 100,60.2,moderate/mixed
1,Debt Service Ratio,2025-10-01,11.32%,27.4,lower
2,Personal Saving Rate,2026-04-01,2.60%,96.4,elevated
3,Credit Card Delinquency Rate,2026-01-01,2.92%,32.6,lower
4,CPI YoY Inflation,2026-05-01,4.27%,72.8,moderate/mixed
5,Real Median Weekly Earnings,2026-01-01,$376,1.1,lower
6,Consumer Sentiment,2026-04-01,49.8 index,99.9,elevated
7,Housing Payment-to-Income Ratio,2026-03-31,28.04%,91.1,elevated


## 2. Current Pressure Drivers

The normalized score converts each household pressure indicator to a 0-100 pressure percentile. Higher bars mean the latest reading is more concerning relative to that indicator's own history. Indicators where lower values are worse, such as saving, real earnings, and sentiment, are reversed.


In [4]:
pressure_tooltip = [
    alt.Tooltip("indicator:N", title="Indicator"),
    alt.Tooltip("description:N", title="What it measures"),
    alt.Tooltip("latest_value:Q", title="Latest raw value", format=".2f"),
    alt.Tooltip("unit:N", title="Unit"),
    alt.Tooltip("latest_date_label:N", title="Latest observation date"),
    alt.Tooltip("pressure_percentile:Q", title="Pressure score", format=".1f"),
    alt.Tooltip("pressure_level:N", title="Pressure level"),
    alt.Tooltip("pressure_direction_label:N", title="Pressure direction"),
]

pressure_base = alt.Chart(household_pressure_summary).encode(
    x=alt.X(
        "pressure_percentile:Q",
        title="Pressure score: 0 = low concern, 100 = high concern",
        scale=alt.Scale(domain=[0, 100]),
    ),
    y=alt.Y(
        "indicator:N",
        title=None,
        sort=alt.SortField(field="pressure_percentile", order="descending"),
    ),
)
pressure_bars = pressure_base.mark_bar(cornerRadiusEnd=3).encode(
    color=alt.Color(
        "pressure_percentile:Q",
        title="Pressure score",
        scale=alt.Scale(domain=[0, 50, 100], range=PRESSURE_COLORS),
    ),
    tooltip=pressure_tooltip,
)
pressure_labels = pressure_base.mark_text(
    align="right",
    baseline="middle",
    dx=-6,
    color="white",
    fontWeight="bold",
).encode(
    text=alt.Text("pressure_percentile:Q", format=".1f")
)
pressure_score_chart = (pressure_bars + pressure_labels).properties(width=780, height=280)
display(pressure_score_chart)


alt.LayerChart(...)

## 3. Household Debt, Saving, and Delinquency Time Series

Each chart shows the full historical series, a dashed historical average, and a point for the latest observation.


In [5]:
def make_indicator_line_chart(series_code, title, y_title, line_color="#33658a"):
    config = SERIES_CONFIG[series_code]
    value_col = config["value_col"]
    df = series_data[series_code]
    latest = df.tail(1)
    average_df = pd.DataFrame({"historical_average": [df[value_col].mean()]})

    line = alt.Chart(df).mark_line(color=line_color, strokeWidth=2).encode(
        x=alt.X("observation_date:T", title="Date"),
        y=alt.Y(f"{value_col}:Q", title=y_title, scale=alt.Scale(zero=False)),
        tooltip=[alt.Tooltip("observation_date:T", title="Date", format="%Y-%m-%d"), alt.Tooltip(f"{value_col}:Q", title="Value (%)", format=".2f")],
    )
    average_rule = alt.Chart(average_df).mark_rule(color="#6c757d", strokeDash=[6, 4], strokeWidth=1.5).encode(
        y=alt.Y("historical_average:Q"),
        tooltip=[alt.Tooltip("historical_average:Q", title="Historical average (%)", format=".2f")],
    )
    latest_point = alt.Chart(latest).mark_circle(size=95, color="#d95f5f", stroke="white", strokeWidth=1).encode(
        x=alt.X("observation_date:T"),
        y=alt.Y(f"{value_col}:Q"),
        tooltip=[alt.Tooltip("observation_date:T", title="Latest date", format="%Y-%m-%d"), alt.Tooltip(f"{value_col}:Q", title="Latest value (%)", format=".2f")],
    )
    return (line + average_rule + latest_point).properties(title=title, width=900, height=240)


### Debt Service Ratio

Debt service shows how much household disposable income goes toward required debt payments. Higher readings leave households with less room to absorb shocks or rising costs.

In [6]:
debt_service_chart = make_indicator_line_chart("TDSP", "Debt Service Ratio", "Debt Service Ratio (%)", line_color="#33658a")
display(debt_service_chart)


alt.LayerChart(...)

### Personal Saving Rate

The personal saving rate is a cushion indicator. Lower saving can mean households have less cash buffer after spending, so low readings increase the pressure score.

In [7]:
saving_rate_chart = make_indicator_line_chart("PSAVERT", "Personal Saving Rate", "Personal Saving Rate (%)", line_color="#2a9d8f")
display(saving_rate_chart)


alt.LayerChart(...)

### Credit Card Delinquency Rate

Credit card delinquency is a direct sign of repayment stress. Rising delinquency suggests more households are falling behind on short-term consumer debt.

In [8]:
delinquency_chart = make_indicator_line_chart("DRCCLACBS", "Credit Card Delinquency Rate", "Credit Card Delinquency Rate (%)", line_color="#9b5de5")
display(delinquency_chart)


alt.LayerChart(...)

## 4. Economic Context Charts

Unemployment and the federal funds rate help explain the environment around households, but they are **not** averaged into the 7-indicator household financial pressure index.


In [9]:
def make_context_chart(series_code, title, y_title, line_color):
    config = SERIES_CONFIG[series_code]
    value_col = config["value_col"]
    df = series_data[series_code]
    latest = df.tail(1)
    line = alt.Chart(df).mark_line(color=line_color, strokeWidth=2).encode(
        x=alt.X("observation_date:T", title="Date"),
        y=alt.Y(f"{value_col}:Q", title=y_title, scale=alt.Scale(zero=False)),
        tooltip=[alt.Tooltip("observation_date:T", title="Date", format="%Y-%m-%d"), alt.Tooltip(f"{value_col}:Q", title="Value (%)", format=".2f")],
    )
    latest_point = alt.Chart(latest).mark_circle(size=90, color="#d95f5f", stroke="white", strokeWidth=1).encode(
        x=alt.X("observation_date:T"), y=alt.Y(f"{value_col}:Q"),
        tooltip=[alt.Tooltip("observation_date:T", title="Latest date", format="%Y-%m-%d"), alt.Tooltip(f"{value_col}:Q", title="Latest value (%)", format=".2f")],
    )
    return (line + latest_point).properties(title=title, width=900, height=240)

unemployment_chart = make_context_chart("UNRATE", "Unemployment Rate", "Unemployment Rate (%)", line_color="#33658a")
fed_funds_chart = make_context_chart("FEDFUNDS", "Federal Funds Rate", "Federal Funds Rate (%)", line_color="#9b5de5")
display(unemployment_chart)
display(fed_funds_chart)


alt.LayerChart(...)

alt.LayerChart(...)

## Additional Household Pressure Indicators

Inflation, real earnings, sentiment, and housing affordability broaden the household pressure framework beyond direct debt and saving measures. These four series are included in the 7-indicator household financial pressure index.


In [10]:
def make_latest_line_chart(df, value_col, title, y_title, line_color="#33658a", value_format=".2f"):
    latest = df.tail(1)
    line = alt.Chart(df).mark_line(color=line_color, strokeWidth=2).encode(
        x=alt.X("observation_date:T", title="Date"),
        y=alt.Y(f"{value_col}:Q", title=y_title, scale=alt.Scale(zero=False)),
        tooltip=[alt.Tooltip("observation_date:T", title="Date", format="%Y-%m-%d"), alt.Tooltip(f"{value_col}:Q", title=y_title, format=value_format)],
    )
    latest_point = alt.Chart(latest).mark_circle(size=90, color="#d95f5f", stroke="white", strokeWidth=1).encode(
        x=alt.X("observation_date:T"), y=alt.Y(f"{value_col}:Q"),
        tooltip=[alt.Tooltip("observation_date:T", title="Latest date", format="%Y-%m-%d"), alt.Tooltip(f"{value_col}:Q", title="Latest value", format=value_format)],
    )
    return (line + latest_point).properties(title=title, width=900, height=240)


### Inflation / CPI Context

Year-over-year CPI inflation captures cost-of-living pressure. Higher inflation can squeeze household budgets even when income or debt measures look stable.

In [11]:
inflation_cpi_yoy_chart = make_latest_line_chart(
    cpi_yoy, "cpi_yoy", "Inflation Pressure: CPI Year-over-Year Change", "CPI year-over-year change (%)", line_color="#d95f5f"
)
display(inflation_cpi_yoy_chart)


alt.LayerChart(...)

### Real Weekly Earnings Context

Real median weekly earnings show whether inflation-adjusted wages are improving or weakening. Lower real earnings count as more pressure in the 7-indicator household framework.


In [12]:
real_weekly_earnings_chart = make_latest_line_chart(
    real_weekly_earnings, "LES1252881600Q", "Real Median Weekly Earnings", "Real median weekly earnings", line_color="#2a9d8f", value_format=",.0f"
)
display(real_weekly_earnings_chart)


alt.LayerChart(...)

### Consumer Sentiment Context

Consumer sentiment captures how households feel about the economy. Lower sentiment can signal more anxiety about income, prices, employment, or future finances.

In [13]:
consumer_sentiment_chart = make_latest_line_chart(
    consumer_sentiment, "UMCSENT", "Consumer Sentiment", "Consumer sentiment index", line_color="#33658a", value_format=".1f"
)
display(consumer_sentiment_chart)


alt.LayerChart(...)

### Housing Affordability Context

This approximation estimates the monthly principal-and-interest payment on the median home sale price using 20% down and a 30-year fixed mortgage, then compares that payment with monthly real median household income. It is part of the 7-indicator household pressure framework.


In [14]:
housing_y_max = max(45, housing_affordability["payment_to_income_pct"].max() * 1.08)
housing_band_order = ["Above 35%: highly strained", "25-35%: strained", "Under 25%: more affordable"]
housing_bands = pd.DataFrame([
    {"band": "Under 25%: more affordable", "start": housing_affordability["observation_date"].min(), "end": housing_affordability["observation_date"].max(), "y_min": 0, "y_max": 25},
    {"band": "25-35%: strained", "start": housing_affordability["observation_date"].min(), "end": housing_affordability["observation_date"].max(), "y_min": 25, "y_max": 35},
    {"band": "Above 35%: highly strained", "start": housing_affordability["observation_date"].min(), "end": housing_affordability["observation_date"].max(), "y_min": 35, "y_max": housing_y_max},
])
housing_bands["band"] = pd.Categorical(housing_bands["band"], categories=housing_band_order, ordered=True)

housing_band_layer = alt.Chart(housing_bands).mark_rect(opacity=0.22).encode(
    x=alt.X("start:T", title="Date"), x2="end:T",
    y=alt.Y("y_min:Q", title="Payment as share of monthly household income (%)"), y2="y_max:Q",
    color=alt.Color("band:N", title="Interpretation band", scale=alt.Scale(domain=housing_band_order, range=["#d95f5f", "#f4a261", "#2a9d8f"])),
    tooltip=[alt.Tooltip("band:N", title="Band")],
)
housing_line = alt.Chart(housing_affordability).mark_line(color="#263238", strokeWidth=2).encode(
    x=alt.X("observation_date:T", title="Date"),
    y=alt.Y("payment_to_income_pct:Q", title="Payment as share of monthly household income (%)", scale=alt.Scale(domain=[0, housing_y_max])),
    tooltip=[
        alt.Tooltip("observation_date:T", title="Quarter", format="%Y-%m-%d"),
        alt.Tooltip("MSPUS:Q", title="Median home sale price", format=",.0f"),
        alt.Tooltip("MORTGAGE30US:Q", title="30-year mortgage rate (%)", format=".2f"),
        alt.Tooltip("MEHOINUSA672N:Q", title="Annual median household income", format=",.0f"),
        alt.Tooltip("estimated_monthly_payment:Q", title="Estimated monthly payment", format=",.0f"),
        alt.Tooltip("payment_to_income_pct:Q", title="Payment-to-income (%)", format=".1f"),
    ],
)
housing_latest_point = alt.Chart(housing_affordability.tail(1)).mark_circle(size=90, color="#d95f5f", stroke="white", strokeWidth=1).encode(
    x=alt.X("observation_date:T"), y=alt.Y("payment_to_income_pct:Q"),
    tooltip=[alt.Tooltip("observation_date:T", title="Latest quarter", format="%Y-%m-%d"), alt.Tooltip("payment_to_income_pct:Q", title="Latest payment-to-income (%)", format=".1f")],
)
housing_affordability_chart = (housing_band_layer + housing_line + housing_latest_point).properties(
    title="Estimated Mortgage Payment as Share of Household Income", width=800, height=260
)
display(housing_affordability_chart)


alt.LayerChart(...)

In [15]:
# Individual source charts are displayed and exported separately; no combined context layout is needed.


## Historical Household Pressure Flags

This timeline counts how many of the 7 household pressure indicators were elevated relative to their own histories in each quarter. Because the series do not all start at the same time, the tooltip includes an available-indicator count for each quarter.


In [16]:
timeline_indicator_sources = {
    "Debt Service Ratio": {
        "df": series_data["TDSP"], "value_col": "TDSP", "timeline_col": "debt_service_ratio",
        "direction": "higher_is_worse", "tooltip_title": "Debt Service Ratio (%)", "tooltip_format": ".2f",
    },
    "Personal Saving Rate": {
        "df": series_data["PSAVERT"], "value_col": "PSAVERT", "timeline_col": "personal_saving_rate",
        "direction": "lower_is_worse", "tooltip_title": "Personal Saving Rate (%)", "tooltip_format": ".2f",
    },
    "Credit Card Delinquency Rate": {
        "df": series_data["DRCCLACBS"], "value_col": "DRCCLACBS", "timeline_col": "credit_card_delinquency_rate",
        "direction": "higher_is_worse", "tooltip_title": "Credit Card Delinquency Rate (%)", "tooltip_format": ".2f",
    },
    "CPI YoY Inflation": {
        "df": cpi_yoy, "value_col": "cpi_yoy", "timeline_col": "cpi_yoy_inflation",
        "direction": "higher_is_worse", "tooltip_title": "CPI YoY Inflation (%)", "tooltip_format": ".2f",
    },
    "Real Median Weekly Earnings": {
        "df": real_weekly_earnings, "value_col": "LES1252881600Q", "timeline_col": "real_weekly_earnings",
        "direction": "lower_is_worse", "tooltip_title": "Real Weekly Earnings", "tooltip_format": ",.0f",
    },
    "Consumer Sentiment": {
        "df": consumer_sentiment, "value_col": "UMCSENT", "timeline_col": "consumer_sentiment",
        "direction": "lower_is_worse", "tooltip_title": "Consumer Sentiment", "tooltip_format": ".1f",
    },
    "Housing Payment-to-Income Ratio": {
        "df": housing_affordability, "value_col": "payment_to_income_pct", "timeline_col": "payment_to_income_pct",
        "direction": "higher_is_worse", "tooltip_title": "Mortgage Payment / Income (%)", "tooltip_format": ".1f",
    },
}

household_pressure_q = None
for indicator in HOUSEHOLD_PRESSURE_INDICATORS:
    source = timeline_indicator_sources[indicator]
    indicator_q = (
        source["df"]
        .set_index("observation_date")[[source["value_col"]]]
        .resample("QE")
        .mean()
        .rename(columns={source["value_col"]: source["timeline_col"]})
    )
    household_pressure_q = indicator_q if household_pressure_q is None else household_pressure_q.join(indicator_q, how="outer")

household_pressure_q = household_pressure_q.reset_index().sort_values("observation_date")
timeline_value_columns = [timeline_indicator_sources[indicator]["timeline_col"] for indicator in HOUSEHOLD_PRESSURE_INDICATORS]
household_pressure_q["available_indicator_count"] = household_pressure_q[timeline_value_columns].notna().sum(axis=1)
household_pressure_q = household_pressure_q[household_pressure_q["available_indicator_count"] > 0].copy()

pressure_flag_columns = []
for indicator in HOUSEHOLD_PRESSURE_INDICATORS:
    source = timeline_indicator_sources[indicator]
    value_col = source["timeline_col"]
    flag_col = f"{value_col}_pressure_flag"
    median_value = household_pressure_q[value_col].median(skipna=True)
    if source["direction"] == "higher_is_worse":
        household_pressure_q[flag_col] = household_pressure_q[value_col] > median_value
    else:
        household_pressure_q[flag_col] = household_pressure_q[value_col] < median_value
    pressure_flag_columns.append(flag_col)

household_pressure_q["pressure_flag_count"] = household_pressure_q[pressure_flag_columns].sum(axis=1)
household_pressure_q["quarter"] = household_pressure_q["observation_date"].dt.to_period("Q").astype(str)

stress_display_start = max(household_pressure_q["observation_date"].min(), pd.Timestamp("1950-01-01"))
stress_display_end = household_pressure_q["observation_date"].max()
stress_x_scale = alt.Scale(domain=[stress_display_start, stress_display_end])
pressure_flags_x = alt.selection_interval(bind="scales", encodings=["x"], name="pressure_flags_x")
household_pressure_display_q = household_pressure_q[
    (household_pressure_q["observation_date"] >= stress_display_start)
    & (household_pressure_q["observation_date"] <= stress_display_end)
].copy()

historical_pressure_tooltip = [
    alt.Tooltip("quarter:N", title="Quarter"),
    alt.Tooltip("pressure_flag_count:Q", title="Pressure flag count", format=".0f"),
    alt.Tooltip("available_indicator_count:Q", title="Available indicators", format=".0f"),
]
historical_pressure_tooltip.extend(
    alt.Tooltip(
        f"{timeline_indicator_sources[indicator]['timeline_col']}:Q",
        title=timeline_indicator_sources[indicator]["tooltip_title"],
        format=timeline_indicator_sources[indicator]["tooltip_format"],
    )
    for indicator in HOUSEHOLD_PRESSURE_INDICATORS
)

historical_pressure_flags_chart = alt.Chart(household_pressure_display_q).mark_bar().encode(
    x=alt.X("observation_date:T", title="Quarter", scale=stress_x_scale, axis=alt.Axis(format="%Y")),
    y=alt.Y("pressure_flag_count:Q", title="Number of household pressure flags", scale=alt.Scale(domain=[0, len(HOUSEHOLD_PRESSURE_INDICATORS)]), axis=alt.Axis(tickMinStep=1)),
    color=alt.Color("pressure_flag_count:Q", title="Pressure flags", scale=alt.Scale(domain=[0, len(HOUSEHOLD_PRESSURE_INDICATORS) / 2, len(HOUSEHOLD_PRESSURE_INDICATORS)], range=["#d7ece8", "#f4a261", "#d95f5f"])),
    tooltip=historical_pressure_tooltip,
).properties(width=900, height=240).add_params(pressure_flags_x)

display(historical_pressure_flags_chart)


alt.Chart(...)

## Pressure Score Logic

`Raw FRED CSV files` -> `clean dates and values` -> `calculate historical percentiles` -> `adjust direction of pressure` -> `average the 7 household pressure indicators` -> `overall household financial pressure index out of 100`

Only the indicators in `HOUSEHOLD_PRESSURE_INDICATORS` flow into the main index. Unemployment and the federal funds rate remain supporting economic context.


## Custom Dashboard Layout

The exported dashboard is organized as a presentation page: current household financial pressure index first, then the current pressure drivers, the historical household pressure flags timeline, housing payment-to-income, and finally an accordion section for supporting/source plots.


In [17]:
from IPython.display import HTML

chart_sections = [
    ("Current Pressure Drivers", "Shows the 7 household pressure indicators ranked by their latest normalized pressure score. Higher scores mean the latest reading is more concerning relative to that indicator's own history.", "normalized_financial_pressure_score.html", "compact"),
    ("Historical Household Pressure Flags", "Counts how many of the 7 household pressure indicators were elevated relative to their own histories in each quarter.", "expanded_household_pressure_flags_timeline.html", "timeline"),
    ("Housing Payment-to-Income Ratio", None, "housing_affordability.html", "housing"),
]

accordion_groups = [
    ("Individual Indicator Source Charts", [
        ("Debt Service Ratio", "debt_service_ratio.html"),
        ("Personal Saving Rate", "personal_saving_rate.html"),
        ("Credit Card Delinquency Rate", "credit_card_delinquency_rate.html"),
        ("CPI Year-over-Year Inflation", "inflation_cpi_yoy.html"),
        ("Real Median Weekly Earnings", "real_weekly_earnings.html"),
        ("Consumer Sentiment", "consumer_sentiment.html"),
        ("Unemployment Rate", "unemployment_rate.html"),
        ("Federal Funds Rate", "federal_funds_rate.html"),
    ]),
]

status_class = overall_status.replace("/", "-").replace(" ", "-")
insight_cards_html = "".join(
    f'''<article class="metric-card">
        <span>{row["Card label"]}</span>
        <h3>{row["Indicator"]}</h3>
        <p>{row["Description"]}</p>
        <strong>{row["Latest value"]}</strong>
        <em>Pressure score: {row["Pressure score"]}<br>Latest observation: {row["Latest observation"]}</em>
    </article>'''
    for _, row in pressure_insight_cards.iterrows()
)
index_driver_summary = household_pressure_summary.assign(
    indicator_name=household_pressure_summary["indicator"].astype(str)
).set_index("indicator_name")
index_driver_cards_html = "".join(
    f'''<article class="index-driver-card">
        <h3>{indicator}</h3>
        <p>{indicator_descriptions[indicator]}</p>
        <em>{index_driver_summary.loc[indicator, "pressure_direction_label"]}.</em>
    </article>'''
    for indicator in HOUSEHOLD_PRESSURE_INDICATORS
)

def chart_iframe(title, description, file_name, variant="standard"):
    note_html = f'<p class="chart-note">{description}</p>' if description else ''
    return f'''<section class="chart-section {variant}">
        <h2>{title}</h2>
        {note_html}
        <iframe src="{file_name}" title="{title}" loading="lazy"></iframe>
    </section>'''

def accordion_html(group_title, charts):
    chart_markup = "".join(
        f'''<div class="accordion-chart">
            <h3>{title}</h3>
            <iframe src="{file_name}" title="{title}" loading="lazy"></iframe>
        </div>'''
        for title, file_name in charts
    )
    return f'''<details class="accordion-item">
        <summary>{group_title}</summary>
        <div class="accordion-body">{chart_markup}</div>
    </details>'''

top_chart_html = "".join(chart_iframe(*frame) for frame in chart_sections)
accordion_markup = "".join(accordion_html(title, charts) for title, charts in accordion_groups)

dashboard_html = f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>U.S. Household Financial Pressure Dashboard</title>
<style>
:root {{
    --ink: #222222;
    --muted: #606975;
    --line: #d9dde3;
    --paper: #f6f7f9;
    --panel: #ffffff;
    --teal: #2a9d8f;
    --accent-blue: #5b9bc8;
    --red: #c84f4f;
    --blue: #33658a;
}}
* {{ box-sizing: border-box; }}
body {{
    margin: 0;
    color: var(--ink);
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
    background: var(--paper);
}}
main {{ width: min(1120px, calc(100% - 32px)); margin: 0 auto; padding: 24px 0 44px; }}
.status-panel {{
    border: 1px solid var(--line);
    border-left: 5px solid var(--accent-blue);
    background: var(--panel);
    padding: 22px;
}}
.status-label {{ margin: 0 0 6px; color: var(--muted); font-size: 13px; font-weight: 600; }}
.status-value {{ margin: 0; font-size: clamp(34px, 5vw, 48px); line-height: 1.05; font-weight: 700; }}
.status-pill {{ display: inline-block; margin-top: 12px; padding: 6px 10px; border: 1px solid #b9d5e8; background: #eef7fc; color: #22536a; font-size: 13px; font-weight: 700; }}
.status-score {{ margin: 12px 0 0; font-size: 15px; color: var(--muted); }}
.status-summary {{ display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 12px; margin-top: 20px; }}
.metric-card {{ border: 1px solid var(--line); background: #ffffff; padding: 14px; min-height: 174px; }}
.metric-card span {{ display: block; color: #2f3740; font-size: 13px; font-weight: 700; line-height: 1.35; }}
.metric-card h3 {{ margin: 8px 0 0; font-size: 17px; line-height: 1.2; }}
.metric-card p {{ margin: 7px 0 0; min-height: 54px; color: var(--muted); font-size: 13px; line-height: 1.4; }}
.metric-card strong {{ display: block; margin: 10px 0 5px; font-size: 24px; line-height: 1.05; }}
.metric-card em {{ color: var(--muted); font-size: 12px; font-style: normal; line-height: 1.4; }}
.index-panel {{ margin-top: 18px; border: 1px solid var(--line); background: var(--panel); padding: 16px; }}
.index-panel h2 {{ margin-bottom: 8px; }}
.index-panel > p {{ margin: 0 0 12px; color: var(--muted); font-size: 14px; line-height: 1.45; }}
.index-driver-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 10px; }}
.index-driver-card {{ border: 1px solid #e6eaee; background: #fff; padding: 12px; }}
.index-driver-card h3 {{ margin: 0 0 6px; color: #2f3740; font-size: 15px; line-height: 1.25; }}
.index-driver-card p {{ margin: 0; color: var(--muted); font-size: 13px; line-height: 1.38; }}
.index-driver-card em {{ display: block; margin-top: 8px; color: #47515d; font-size: 12px; font-style: normal; line-height: 1.35; }}
.chart-section, .accordion-panel {{ margin-top: 18px; border: 1px solid var(--line); background: var(--panel); padding: 16px; }}
.chart-section.timeline {{ overflow-x: auto; -webkit-overflow-scrolling: touch; }}
h2 {{ margin: 0 0 12px; font-size: 20px; font-weight: 700; }}
.chart-note {{ margin: -4px 0 12px; color: var(--muted); font-size: 14px; line-height: 1.45; }}
h3 {{ margin: 0 0 10px; font-size: 16px; font-weight: 700; }}
iframe {{ width: 100%; border: 0; background: #ffffff; }}
.chart-section.compact iframe {{ height: 400px; }}
.chart-section.housing iframe {{ height: 360px; }}
.chart-section.timeline iframe {{ height: 400px; min-width: 900px; }}
.accordion-panel > p {{ margin: -4px 0 16px; color: var(--muted); font-size: 14px; line-height: 1.5; }}
.accordion-item {{ border-top: 1px solid var(--line); background: #ffffff; }}
.accordion-item:last-child {{ border-bottom: 1px solid var(--line); }}
summary {{ cursor: pointer; padding: 14px 4px; font-size: 15px; font-weight: 700; line-height: 1.2; }}
.accordion-body {{ display: grid; gap: 18px; padding: 0 0 18px; }}
.accordion-chart {{ border: 1px solid #e6eaee; padding: 14px; background: #fff; overflow-x: auto; -webkit-overflow-scrolling: touch; }}
.accordion-chart iframe {{ height: 380px; min-width: 900px; }}
@media (max-width: 860px) {{
    .status-panel {{ padding: 18px; }}
    .status-summary {{ grid-template-columns: 1fr; margin-top: 18px; }}
    .index-driver-grid {{ grid-template-columns: 1fr; }}
    main {{ width: min(100% - 20px, 1180px); padding-top: 12px; }}
    .index-panel, .chart-section, .accordion-panel {{ padding: 12px; }}
    .chart-section.compact iframe, .chart-section.housing iframe, .accordion-chart iframe {{ height: 420px; }}
    .chart-section.timeline iframe {{ height: 470px; min-width: 900px; }}
}}
</style>
</head>
<body>
<main>
    <section class="status-panel">
        <div>
            <p class="status-label">Current household financial pressure index</p>
            <h1 class="status-value">{overall_status.title()}</h1>
            <span class="status-pill {status_class}">{overall_pressure_score:.1f} / 100</span>
            <p class="status-score">Based on each indicator's latest available FRED observation.</p>
        </div>
        <div class="status-summary">{insight_cards_html}</div>
    </section>
    <section class="index-panel">
        <h2>What’s in the index?</h2>
        <p>The household financial pressure index is built from seven drivers. Each driver is converted into a normalized 0-100 pressure score based on its own history, so higher scores mean the latest reading is more concerning for that series.</p>
        <div class="index-driver-grid">{index_driver_cards_html}</div>
    </section>
    {top_chart_html}
    <section class="accordion-panel">
        <h2>Individual Source Charts</h2>
        <p>Open the accordion to inspect the individual source series behind the index and supporting context.</p>
        {accordion_markup}
    </section>
</main>
</body>
</html>'''

display(HTML(dashboard_html))



## Save Outputs

The notebook saves the main dashboard shell plus standalone chart files. The main dashboard uses the standalone files as embedded panels, including accordions for the supporting individual plots.


In [18]:
scorecard_display_for_output = scorecard_display.rename(columns={"Pressure percentile": "Normalized pressure score"})

scorecard_html = f'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><title>Household Financial Pressure Scorecard</title>
<style>
body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; margin: 32px; color: #202124; }}
h1 {{ font-size: 24px; margin-bottom: 6px; }}
p {{ max-width: 900px; line-height: 1.45; color: #4b5563; }}
table {{ border-collapse: collapse; min-width: 880px; margin-top: 18px; }}
th, td {{ border-bottom: 1px solid #d9dee3; padding: 10px 12px; text-align: left; }}
th {{ background: #f4f6f8; font-weight: 650; }}
tr:first-child td {{ font-weight: 650; background: #fbfcfd; }}
</style></head><body>
<h1>U.S. Household Financial Pressure Scorecard</h1>
<p>The overall score averages the seven household normalized pressure scores in HOUSEHOLD_PRESSURE_INDICATORS. The score uses each indicator's latest available FRED observation, so individual dates differ by release schedule.</p>
{scorecard_display_for_output.to_html(index=False, escape=False)}
</body></html>'''

selected_outputs = {
    "normalized_financial_pressure_score.html": pressure_score_chart,
    "expanded_household_pressure_flags_timeline.html": historical_pressure_flags_chart,
    "debt_service_ratio.html": debt_service_chart,
    "personal_saving_rate.html": saving_rate_chart,
    "credit_card_delinquency_rate.html": delinquency_chart,
    "unemployment_rate.html": unemployment_chart,
    "federal_funds_rate.html": fed_funds_chart,
    "inflation_cpi_yoy.html": inflation_cpi_yoy_chart,
    "real_weekly_earnings.html": real_weekly_earnings_chart,
    "consumer_sentiment.html": consumer_sentiment_chart,
    "housing_affordability.html": housing_affordability_chart,
}

(OUTPUT_DIR / "overview_scorecard.html").write_text(scorecard_html, encoding="utf-8")
(OUTPUT_DIR / "financial_pressure_dashboard.html").write_text(dashboard_html, encoding="utf-8")
for filename, chart in selected_outputs.items():
    chart.save(OUTPUT_DIR / filename)

print("Saved selected HTML outputs:")
for filename in ["financial_pressure_dashboard.html", "overview_scorecard.html", *selected_outputs.keys()]:
    print(OUTPUT_DIR / filename)


Saved selected HTML outputs:
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/financial_pressure_dashboard.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/overview_scorecard.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/normalized_financial_pressure_score.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/expanded_household_pressure_flags_timeline.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/debt_service_ratio.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/personal_saving_rate.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/credit_card_delinquency_rate.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/unemployment_rate.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/federal_funds_rate.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_

In [19]:
from IPython.display import Markdown

biggest_insight = pressure_insight_cards.loc[pressure_insight_cards["Card label"] == "Biggest pressure driver"].iloc[0]
broadest_insight = pressure_insight_cards.loc[pressure_insight_cards["Card label"] == "Broadest household strain signal"].iloc[0]
lowest_insight = pressure_insight_cards.loc[pressure_insight_cards["Card label"] == "Lowest current pressure signal"].iloc[0]

display(Markdown(f"""
## Written Interpretation

The dashboard uses a 7-indicator household financial pressure index. The current index is **{overall_pressure_score:.1f} out of 100**, which is **{overall_status}**, based on each indicator's latest available FRED observation.

The top insight cards are dynamically selected from the latest pressure percentiles and avoid duplicate indicators when possible. The **biggest pressure driver** is **{biggest_insight['Indicator']}** at **{biggest_insight['Pressure score']}**. The **broadest household strain signal** is **{broadest_insight['Indicator']}** at **{broadest_insight['Pressure score']}**, selected from consumer sentiment and housing payment burden after excluding the biggest driver when another candidate is available. The **lowest current pressure signal** is **{lowest_insight['Indicator']}** at **{lowest_insight['Pressure score']}**.

The historical household pressure flags chart shows how many of the 7 household pressure indicators were elevated relative to their own histories at the same time in each quarter. Higher counts mean pressure was broader across the household framework, while the tooltip shows how many of the 7 indicators were available in that quarter.

Unemployment and the federal funds rate remain supporting economic context only; they are not included in the 7-indicator household financial pressure index.

The main HTML file to open is **`html_charts/financial_pressure_dashboard.html`**.
"""))



## Written Interpretation

The dashboard uses a 7-indicator household financial pressure index. The current index is **60.2 out of 100**, which is **moderate/mixed**, based on each indicator's latest available FRED observation.

The top insight cards are dynamically selected from the latest pressure percentiles and avoid duplicate indicators when possible. The **biggest pressure driver** is **Consumer Sentiment** at **99.9 / 100**. The **broadest household strain signal** is **Housing Payment-to-Income Ratio** at **91.1 / 100**, selected from consumer sentiment and housing payment burden after excluding the biggest driver when another candidate is available. The **lowest current pressure signal** is **Real Median Weekly Earnings** at **1.1 / 100**.

The historical household pressure flags chart shows how many of the 7 household pressure indicators were elevated relative to their own histories at the same time in each quarter. Higher counts mean pressure was broader across the household framework, while the tooltip shows how many of the 7 indicators were available in that quarter.

Unemployment and the federal funds rate remain supporting economic context only; they are not included in the 7-indicator household financial pressure index.

The main HTML file to open is **`html_charts/financial_pressure_dashboard.html`**.
